<a href="https://www.kaggle.com/code/nicapotato/sparse-n-dense-logistic-n-tsvd-starter?scriptVersionId=320612038" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Logistic Regression Starter
_By Nick Brooks_

**Content:** <br>
- Bag of Words: Term-Frequency Inverse Document Frequency Method
- Sparse Matrix with "Dense Features"
- Logistic Regression
- Logistic Regression and Truncated Singular Value Decomposition

## Packages and Load

In [1]:
import time
notebookstart= time.time()

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
import gc
print("Data:\n",os.listdir("../input"))

# Models Packages
from sklearn import metrics
from sklearn.metrics import mean_squared_error
from sklearn import feature_selection
from sklearn.model_selection import train_test_split
from sklearn import preprocessing
from sklearn.preprocessing import MinMaxScaler

# Logistic Regression
from sklearn.linear_model import LogisticRegression

# Dimensionality Reduction
from sklearn.decomposition import TruncatedSVD

# Tf-Idf
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from scipy.sparse import hstack, csr_matrix, vstack
from nltk.corpus import stopwords

# Viz
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display

Data:
 ['sampleSubmission.csv', 'train.tsv.zip', 'test.tsv.zip']


In [2]:
df = pd.read_csv("/kaggle/input/train.tsv.zip", sep="\t", index_col = ["PhraseId"])#.sample(500) # Debugging..
trainlen = df.shape[0]
test_df = pd.read_csv("/kaggle/input/test.tsv.zip", sep="\t", index_col = ["PhraseId"])#.sample(500)
testdex = test_df.index
print("\nTrain Shape: ",df.shape)
print("Test Shape: ",test_df.shape)

y = df.Sentiment.copy()
df = pd.concat([df.drop("Sentiment",axis=1),test_df], axis=0)
print("All Data Shape: {} Rows, {} Columns".format(*df.shape))
del test_df

# Glimpse at Dataset 
print("Dataset Glimpse")
display(df.head(5))


Train Shape:  (156060, 3)
Test Shape:  (66292, 2)
All Data Shape: 222352 Rows, 2 Columns
Dataset Glimpse


,SentenceId,Phrase
PhraseId,,
1,1,A series of escapades demonstrating the adage ...
2,1,A series of escapades demonstrating the adage ...
3,1,A series
4,1,A
5,1,series


**Dependent Variable Class Distribution:** <br>

In [3]:
print("Percent Representation by Sentiment Level")
print(y.value_counts(normalize=True)*100)

Percent Representation by Sentiment Level
2    50.994489
3    21.098936
1    17.475971
4     5.899013
0     4.531590
Name: Sentiment, dtype: float64


These classes are imbalanced. Approaches such as stratification and class weights are routes to improve the model.

## Text Features
Here we have some meta text features, what I characterize as the higher level imformation about the language.

In [4]:
# Meta Text Features
df["Phrase"] = df["Phrase"].astype(str) 
df["Phrase"] = df["Phrase"].astype(str).fillna('missing') # FILL NA
df["Phrase"] = df["Phrase"].str.lower() # Lowercase all text, so that capitalized words dont get treated differently
df["Phrase" + '_num_words'] = df["Phrase"].apply(lambda comment: len(comment.split())) # Count number of Words
df["Phrase" + '_num_unique_words'] = df["Phrase"].apply(lambda comment: len(set(w for w in comment.split())))
df["Phrase" + '_words_vs_unique'] = df["Phrase"+'_num_unique_words'] / df["Phrase"+'_num_words'] * 100 # Count Unique Words

## Term Frequence - Inverse Document Frequency
While a simple bag of words method might just count the occurence of each word in each sample and one hot encode, TF-IDF weights how defining (or rare) a certain word in a sample is, in comparison with the rest of the samples.

In [5]:
word_vectorizer = TfidfVectorizer(
    sublinear_tf=True,
    strip_accents='unicode',
    analyzer='word',
    token_pattern=r'\w{1,}',
    stop_words='english',
    ngram_range=(1, 1),
    dtype = np.float32,
    norm='l2',
    min_df=0,
    smooth_idf=False,
    max_features=15000)
# Fit and Transform
word_vectorizer.fit(df.iloc[0:trainlen,:]["Phrase"])
train_word_features = word_vectorizer.transform(df.iloc[0:trainlen,:]["Phrase"])
test_word_features = word_vectorizer.transform(df.iloc[trainlen:,:]["Phrase"])

**Dummy Variable the Sentence Id** <br>
While some models can get away with interger encoding categorical variables, logistic regression would not pick up on the category without dummy encoding.

In [6]:
sent_dummy = pd.get_dummies(df["SentenceId"])
df.drop("SentenceId", axis=1, inplace=True)

**Scale Data Down:** <br>
Below there is a table with the descriptive statistics for my variables. Since most of my other features are on around 0 and 1, I will reduce the magnitude of these features so that the logistic regression will converge better and faster.

In [7]:
print("Before..")
display(df.describe())
dense_variables = [x for x in df.columns if x not in ["PhraseId","SentenceId","Phrase"]]
scaler = MinMaxScaler()
df[dense_variables] = scaler.fit_transform(df[dense_variables])
print("After..")
display(df.describe())

Before..


,Phrase_num_words,Phrase_num_unique_words,Phrase_words_vs_unique
count,222352.000000,222352.000000,222350.000000
mean,7.046908,6.692591,97.861841
std,6.954884,6.261226,5.213006
min,0.000000,0.000000,28.571429
25%,2.000000,2.000000,100.000000
50%,4.000000,4.000000,100.000000
75%,9.000000,9.000000,100.000000
max,56.000000,47.000000,100.000000


After..


,Phrase_num_words,Phrase_num_unique_words,Phrase_words_vs_unique
count,222352.000000,222352.000000,222350.000000
mean,0.125838,0.142396,0.970066
std,0.124194,0.133218,0.072982
min,0.000000,0.000000,0.000000
25%,0.035714,0.042553,1.000000
50%,0.071429,0.085106,1.000000
75%,0.160714,0.191489,1.000000
max,1.000000,1.000000,1.000000


**Fill Missing Values with 0**

In [8]:
# Fill Missing Values with 0
print("Missing Values Before:\n", df.isnull().sum())
df.fillna(0,inplace=True)

Missing Values Before:
 Phrase                     0
Phrase_num_words           0
Phrase_num_unique_words    0
Phrase_words_vs_unique     2
dtype: int64


## Sparse Matrix for Modeling
Sparcity refers to the data storage structure. Instead of having to explicitly use memory to assign each cell of a value, the sparse matrix assumes a matrix of zeros so that it only needs to declare where there is a none-zero value, which only represents 0.00029 % for my processed data for modeling.

In [9]:
# Sparse Matrix
dense_vars = [x for x in df.columns if x not in ["PhraseId","SentenceId","Phrase"]]
X = hstack([csr_matrix(df.iloc[0:trainlen,:][dense_vars].values), csr_matrix(sent_dummy.iloc[0:trainlen,:]),train_word_features])
test_df = hstack([csr_matrix(df.iloc[trainlen:,:][dense_vars].values),csr_matrix(sent_dummy.iloc[trainlen:,:]), test_word_features])
# del df, sent_dummy, train_word_features, test_word_features; gc.collect();
# Zero Proportion
zero_proportion = ((X.toarray() != 0).sum() / (X.shape[0]*X.shape[1]))
print("Portion of Data that has an value other than 0: {}%".format(round(zero_proportion, 5)))

Portion of Data that has an value other than 0: 0.00029%


## Data Ready for Supervised Learning!
Notice the amount of features! 

In [10]:
print("Train Shape: {} Rows and {} Cols".format(*X.shape))
print("Test Shape: {} Rows and {} Cols".format(*test_df.shape))

Train Shape: 156060 Rows and 26830 Cols
Test Shape: 66292 Rows and 26830 Cols


## Logistic Regression <br>
A more traditional classification model in supervised learning. Serves as a great baseline model because of its simplicity, interpretability, and our experience with it as a human race.

In [11]:
# Define and Fit
model = LogisticRegression(multi_class = 'ovr', C=1, solver='sag')
model.fit(X,y)

LogisticRegression(C=1, class_weight=None, dual=False, fit_intercept=True,
          intercept_scaling=1, max_iter=100, multi_class='ovr', n_jobs=1,
          penalty='l2', random_state=None, solver='sag', tol=0.0001,
          verbose=0, warm_start=False)

**Predict and Submit:**

In [12]:
# Predict and Submit
submission = model.predict(test_df)
submission_df = pd.Series(submission).rename("Sentiment")
submission_df.index = testdex
submission_df.to_csv("Logistic_sub.csv",index=True,header=True)
display(submission_df.head())

del model, submission, submission_df

PhraseId
156061    3
156062    3
156063    2
156064    3
156065    3
Name: Sentiment, dtype: int64

## Add Dimensionality Reduction - Truncated Singular Value Decomposition

In [13]:
# Define and Fit TruncatedSVD Dimensionality Reduction Model
svd = TruncatedSVD(n_components=50, n_iter=20, random_state=42)
svd.fit(X) 

# Transform
X = svd.transform(X)
test_df = svd.transform(test_df)

# Logistic
model = LogisticRegression(multi_class = 'ovr', C=1)
model.fit(X,y)

# Submit
submission = model.predict(test_df)
submission_df = pd.Series(submission).rename("Sentiment")
submission_df.index = testdex
submission_df.to_csv("TSVD_n_Logistic_sub.csv",index=True,header=True)
submission_df.head()

PhraseId
156061    2
156062    2
156063    2
156064    2
156065    2
Name: Sentiment, dtype: int64

In [14]:
print("Notebook Runtime: %0.0f seconds"%((time.time() - notebookstart)))

Notebook Runtime: 111 seconds
